# KolektorSDD2 — data exploration

Phase 1 audit only. No model is trained here.

This notebook reads the report produced by `scripts/run_data_audit.py`.
Re-run that script if `reports/exploration/audit_summary.json` is missing.

Problem statement: see `PROBLEM.md`.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, Markdown, display

ROOT = Path("..").resolve()
REPORT_DIR = ROOT / "reports" / "exploration"
report = json.loads((REPORT_DIR / "audit_summary.json").read_text(encoding="utf-8"))
print("Generated UTC:", report["generated_at_utc"])
print("Samples:", report["n_samples"])

## Sample counts and class imbalance

Labels are derived from official masks: any nonzero ground-truth pixel → `defective`, else `ok`.

In [ ]:
rows = []
for split in ("train", "test", "all"):
    c = report["counts"][split]
    imb = report["imbalance"][split]
    rows.append(
        {
            "split": split,
            "n": c["n"],
            "defective": c["defective"],
            "ok": c["ok"],
            "defective_pct": round(100 * imb["defective_fraction"], 2),
            "ok_per_defective": round(imb["ok_to_defective"], 2),
        }
    )
display(rows)
official = report["official_published"]
match = (
    report["counts"]["all"]["n"] == official["images"]
    and report["counts"]["train"]["defective"] == official["train_defective"]
    and report["counts"]["test"]["ok"] == official["test_ok"]
)
print("Matches official published counts:", match)

## Image dimensions and channels

In [ ]:
geo = report["geometry"]
print(f"Width: {geo['width_min']}–{geo['width_max']} px (median {geo['width_median']})")
print(f"Height: {geo['height_min']}–{geo['height_max']} px (median {geo['height_median']})")
print("Modes:", geo["modes"])
print("Channels:", geo["channels"])
print("Distinct sizes:", geo["unique_size_count"])
print("Most common sizes:")
display(geo["most_common_sizes"])

## Integrity: missing, corrupt, unexpected files

In [ ]:
print("Missing masks:", report["missing_masks"])
print("Unreadable images/masks:", len(report["corrupt"]))
print("Unexpected extra files:", report["unexpected_files"])
print(
    "The two copy files are in the official zip and are byte-identical to train/10301. "
    "They are excluded from the canonical inventory."
)

## Defect mask size

Computed only on images labeled defective.

In [ ]:
masks = report["masks"]
print(
    f"Foreground pixels: min {masks['foreground_pixels_min']}, "
    f"median {masks['foreground_pixels_median']}, max {masks['foreground_pixels_max']}"
)
print(
    f"Area fraction: min {masks['area_fraction_min']:.6f}, "
    f"median {masks['area_fraction_median']:.4f}, "
    f"mean {masks['area_fraction_mean']:.4f}, max {masks['area_fraction_max']:.4f}"
)
print("Defects < 0.5% of image:", masks["tiny_defects_under_0_5pct"])
print("Defects < 0.1% of image:", masks["tiny_defects_under_0_1pct"])

## Duplicates and near-duplicates

In [ ]:
exact = report["exact_duplicates"]
near = report["near_duplicates"]
print("SHA-256 duplicate groups:", exact["duplicate_groups"])
print("SHA-256 cross-split groups:", exact["cross_split_groups"])
print("Identical 8x8 aHash groups:", near["identical_ahash_groups"])
print("Images in those groups:", near["images_in_identical_ahash_groups"])
if "close_pairs" in near:
    print(
        "Hamming 1–4 pairs (too coarse to use as leakage):",
        near["close_pairs"],
        "cross-split",
        near["close_pairs_cross_split"],
    )
print("Identical-aHash examples:")
display(near["identical_ahash_examples"])

## Sample images

Grids written by the audit script (random seed 0, plus smallest/largest defect masks).

In [ ]:
for name in [
    "samples_train_ok.png",
    "samples_train_defective.png",
    "samples_test_ok.png",
    "samples_test_defective.png",
    "samples_smallest_defects.png",
    "samples_largest_defects.png",
]:
    path = REPORT_DIR / name
    display(Markdown(f"**{name}**"))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing", path, "— re-run scripts/run_data_audit.py")

## Train / validation / test considerations

- Official test (1,004 images) stays frozen.
- Validation must be carved from official train only, stratified by `defective` / `ok`.
- Exclude `train/10301 (copy).png` and its mask. Canonical inventory already does this.
- Do not use test masks or test-derived statistics to choose a threshold.
- Images are all RGB but not one size (601 distinct shapes). Phase 2 must pick a resize that does not wipe out sub-0.1% defects.
- Prevalence is ~10.7% defective. Accuracy is not the primary metric; see `PROBLEM.md`.